# 📝 Testing (pytest, unittest, mocking)
### Exercises & Solutions — 28 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- unittest basics: TestCase, assertions, setUp/tearDown (1-5)
- pytest basics: plain asserts, raises, approx (6-9)
- Fixtures: basic, scoped, parametrized, yield-based teardown (10-14)
- Parametrize for data-driven tests (15-17)
- Mocking: Mock, MagicMock, patch, side_effect, spec (18-25)
- Real-world test patterns: testing exceptions, async, fixtures chains (26-28)


---


### 1. unittest.TestCase with Multiple Assertion Types

Write a `TestCase` exercising `assertEqual`, `assertTrue`, `assertIn`, `assertIsNone`, and `assertIsInstance` against a simple `Inventory` class.

In [ ]:
import unittest

class Inventory:
    def __init__(self):
        self.items = {}
    def add(self, name, qty):
        self.items[name] = qty
    def get(self, name):
        return self.items.get(name)

class TestInventory(unittest.TestCase):
    def setUp(self):
        self.inv = Inventory()
        self.inv.add("apples", 10)

    def test_assertions(self):
        self.assertEqual(self.inv.get("apples"), 10)
        self.assertTrue("apples" in self.inv.items)
        self.assertIn("apples", self.inv.items)
        self.assertIsNone(self.inv.get("bananas"))
        self.assertIsInstance(self.inv.items, dict)

suite = unittest.TestLoader().loadTestsFromTestCase(TestInventory)
unittest.TextTestRunner(verbosity=2).run(suite)

### 2. setUp / tearDown Lifecycle with Resource Tracking

Write a `TestCase` using `setUp`/`tearDown` to create and clean up a temp file, proving teardown ALWAYS runs (even verify it ran via a flag).

In [ ]:
import unittest, tempfile, os

class TestFileHandling(unittest.TestCase):
    def setUp(self):
        self.tmpfile = tempfile.mktemp()
        with open(self.tmpfile, "w") as f:
            f.write("test data")
        TestFileHandling.setup_ran = True

    def tearDown(self):
        os.remove(self.tmpfile)
        TestFileHandling.teardown_ran = True

    def test_file_exists(self):
        self.assertTrue(os.path.exists(self.tmpfile))

suite = unittest.TestLoader().loadTestsFromTestCase(TestFileHandling)
unittest.TextTestRunner(verbosity=0).run(suite)
print("setUp ran:", TestFileHandling.setup_ran)
print("tearDown ran:", TestFileHandling.teardown_ran)

### 3. assertRaises as Context Manager vs Direct Call

Show BOTH styles of `assertRaises`: as a context manager (to inspect the exception) and the direct-call style (`assertRaises(Exc, func, *args)`).

In [ ]:
import unittest

def withdraw(balance, amount):
    if amount > balance:
        raise ValueError(f"Insufficient funds: balance={balance}, requested={amount}")
    return balance - amount

class TestWithdraw(unittest.TestCase):
    def test_context_manager_style(self):
        with self.assertRaises(ValueError) as ctx:
            withdraw(100, 200)
        self.assertIn("Insufficient", str(ctx.exception))

    def test_direct_call_style(self):
        self.assertRaises(ValueError, withdraw, 100, 200)

suite = unittest.TestLoader().loadTestsFromTestCase(TestWithdraw)
unittest.TextTestRunner(verbosity=2).run(suite)

### 4. setUpClass / tearDownClass for Expensive Shared Setup

Use `setUpClass`/`tearDownClass` (class-level, run ONCE for all tests) to simulate an expensive shared resource like a DB connection, vs per-test `setUp`.

In [ ]:
import unittest

class TestExpensiveResource(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.shared_connection = "DB_CONNECTION_OBJECT"
        cls.setup_count = getattr(cls, "setup_count", 0) + 1
        print("setUpClass ran (expensive, should happen ONCE)")

    @classmethod
    def tearDownClass(cls):
        print("tearDownClass ran (cleanup, ONCE)")

    def test_one(self):
        self.assertEqual(self.shared_connection, "DB_CONNECTION_OBJECT")

    def test_two(self):
        self.assertEqual(self.shared_connection, "DB_CONNECTION_OBJECT")

suite = unittest.TestLoader().loadTestsFromTestCase(TestExpensiveResource)
unittest.TextTestRunner(verbosity=0).run(suite)
print(f"setUpClass call count: {TestExpensiveResource.setup_count} (should be 1, not 2)")

### 5. Skipping Tests Conditionally

Use `@unittest.skip` and `@unittest.skipIf` to conditionally skip tests based on a runtime condition (e.g. platform or feature flag).

In [ ]:
import unittest, sys

FEATURE_ENABLED = False

class TestConditional(unittest.TestCase):
    @unittest.skip("Not implemented yet")
    def test_always_skipped(self):
        self.fail("should never run")

    @unittest.skipIf(not FEATURE_ENABLED, "Feature flag disabled")
    def test_feature_gated(self):
        self.fail("should be skipped since FEATURE_ENABLED=False")

    def test_normal(self):
        self.assertTrue(True)

suite = unittest.TestLoader().loadTestsFromTestCase(TestConditional)
unittest.TextTestRunner(verbosity=2).run(suite)

### 6. pytest Plain Assert Statements

Write pytest-style test functions using plain `assert` (no special methods needed) covering equality, membership, and boolean checks.

In [ ]:
def calculate_bmi(weight_kg, height_m):
    return weight_kg / (height_m ** 2)

def test_bmi_calculation():
    assert calculate_bmi(70, 1.75) == 70 / (1.75 ** 2)

def test_bmi_is_positive():
    result = calculate_bmi(70, 1.75)
    assert result > 0

def test_bmi_membership():
    categories = ["underweight", "normal", "overweight", "obese"]
    assert "normal" in categories

test_bmi_calculation()
test_bmi_is_positive()
test_bmi_membership()
print("All plain-assert tests passed")

### 7. pytest.raises with match= for Message Validation

Use `pytest.raises(ExceptionType, match=regex)` to verify BOTH the exception type AND that its message matches a pattern.

In [ ]:
import pytest

def parse_age(value):
    age = int(value)
    if age < 0:
        raise ValueError(f"Age cannot be negative: {age}")
    return age

def test_negative_age_raises_with_message():
    with pytest.raises(ValueError, match="cannot be negative"):
        parse_age(-5)

def test_invalid_string_raises_value_error():
    with pytest.raises(ValueError):
        parse_age("not a number")

test_negative_age_raises_with_message()
test_invalid_string_raises_value_error()
print("Exception-matching tests passed")

### 8. pytest.approx for Floating-Point Comparisons

Demonstrate why direct `==` comparison fails for floats due to precision, and how `pytest.approx` fixes it.

In [ ]:
import pytest

def test_float_equality_fails_naively():
    result = 0.1 + 0.2
    assert result != 0.3              # proves floating point imprecision is real!
    print(f"0.1 + 0.2 = {result!r} (NOT exactly 0.3)")

def test_float_equality_with_approx():
    result = 0.1 + 0.2
    assert result == pytest.approx(0.3)    # this correctly passes

test_float_equality_fails_naively()
test_float_equality_with_approx()
print("approx-based comparison passed correctly")

### 9. Testing Warnings with pytest.warns

Write a function that emits a `DeprecationWarning`, then test for it using `pytest.warns`.

In [ ]:
import warnings, pytest

def old_function():
    warnings.warn("old_function is deprecated, use new_function instead", DeprecationWarning)
    return 42

def test_deprecation_warning_raised():
    with pytest.warns(DeprecationWarning, match="deprecated"):
        result = old_function()
    assert result == 42

test_deprecation_warning_raised()
print("Warning-detection test passed")

### 10. Basic Fixture for Reusable Test Setup

Define a pytest fixture `sample_data()` returning a reusable test dataset, and TWO test functions consuming it independently.

In [ ]:
import pytest

@pytest.fixture
def sample_data():
    return {"users": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]}

def test_user_count(sample_data):
    assert len(sample_data["users"]) == 2

def test_first_user_name(sample_data):
    assert sample_data["users"][0]["name"] == "Alice"

# Demonstrating manually since pytest fixtures need the pytest runner:
data = sample_data.__wrapped__() if hasattr(sample_data, "__wrapped__") else {"users": [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]}
print("Fixture data:", data)
print("(In a real pytest run, fixtures are injected automatically by name-matching)")

### 11. Fixture Scopes: function vs module vs session

Demonstrate (via simulated counters) how `scope='function'` re-runs setup every test, while `scope='module'` runs ONCE for all tests in that file.

In [ ]:
call_counts = {"function_scope": 0, "module_scope": 0}

def function_scoped_fixture():
    call_counts["function_scope"] += 1
    return "fresh instance"

_module_cache = {}
def module_scoped_fixture():
    if "instance" not in _module_cache:
        call_counts["module_scope"] += 1
        _module_cache["instance"] = "shared instance"
    return _module_cache["instance"]

# Simulate 3 "tests" each requesting both fixtures
for i in range(3):
    function_scoped_fixture()
    module_scoped_fixture()

print(f"function-scoped setup ran: {call_counts['function_scope']} times (once per test)")
print(f"module-scoped setup ran: {call_counts['module_scope']} times (once for the whole module)")

### 12. Fixture with yield for Setup AND Teardown

Build a fixture using `yield` (not `return`) so code AFTER the yield runs as teardown, demonstrated with a resource-tracking pattern.

In [ ]:
import pytest

events = []

@pytest.fixture
def tracked_resource():
    events.append("setup")
    resource = {"connected": True}
    yield resource              # this value is what gets injected into the test
    events.append("teardown")   # runs AFTER the test completes, even if it fails
    resource["connected"] = False

# Manually simulating fixture lifecycle (pytest does this automatically per-test)
gen = tracked_resource.__wrapped__()
resource = next(gen)
print("During test:", resource, events)
try:
    next(gen)
except StopIteration:
    pass
print("After test:", resource, events)

### 13. Fixture Dependency Chains

Build 3 chained fixtures (`db` → `seeded_db` → `repository`) where each depends on the previous, demonstrating pytest's automatic dependency wiring.

In [ ]:
import pytest

@pytest.fixture
def db():
    return {"tables": {}}

@pytest.fixture
def seeded_db(db):
    db["tables"]["users"] = [{"id": 1, "name": "Alice"}]
    return db

@pytest.fixture
def repository(seeded_db):
    class Repo:
        def __init__(self, database): self.db = database
        def get_user(self, uid):
            return next(u for u in self.db["tables"]["users"] if u["id"] == uid)
    return Repo(seeded_db)

# Manually resolving the chain to demonstrate the wiring (pytest does this automatically)
_db = db.__wrapped__()
_seeded = seeded_db.__wrapped__(_db)
_repo = repository.__wrapped__(_seeded)
print(_repo.get_user(1))

### 14. Fixture with Finalizer via request.addfinalizer

Show the alternative teardown mechanism `request.addfinalizer()` (older style, still valid) vs the modern `yield`-based approach.

In [ ]:
cleanup_log = []

class FakeRequest:
    """Simulates pytest's request fixture for demonstration purposes."""
    def __init__(self):
        self._finalizers = []
    def addfinalizer(self, fn):
        self._finalizers.append(fn)
    def run_finalizers(self):
        for fn in reversed(self._finalizers):    # LIFO order, like real pytest
            fn()

def resource_fixture(request):
    cleanup_log.append("resource created")
    def cleanup():
        cleanup_log.append("resource cleaned up")
    request.addfinalizer(cleanup)
    return "resource"

req = FakeRequest()
resource = resource_fixture(req)
print("Resource:", resource)
req.run_finalizers()    # pytest calls this automatically when the test finishes
print("Cleanup log:", cleanup_log)

### 15. Basic Parametrize for Multiple Input/Output Pairs

Use `@pytest.mark.parametrize` to test a `is_leap_year(year)` function against 6 known cases in ONE test function definition.

In [ ]:
import pytest

def is_leap_year(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

@pytest.mark.parametrize("year,expected", [
    (2024, True), (2023, False), (2000, True), (1900, False), (2100, False), (2400, True),
])
def test_is_leap_year(year, expected):
    assert is_leap_year(year) == expected

# Manually run all cases to demonstrate (pytest runs each as a separate test)
for year, expected in [(2024, True), (2023, False), (2000, True), (1900, False), (2100, False), (2400, True)]:
    result = is_leap_year(year)
    status = "PASS" if result == expected else "FAIL"
    print(f"{status}: is_leap_year({year}) = {result} (expected {expected})")

### 16. Parametrize with Multiple Parameters and IDs

Use parametrize with TWO independent parameter sets (creating a cartesian product of test cases) and custom test IDs for readability.

In [ ]:
import pytest

def apply_discount(price, pct):
    return round(price * (1 - pct/100), 2)

@pytest.mark.parametrize("price", [100, 50, 200], ids=["price100", "price50", "price200"])
@pytest.mark.parametrize("pct", [0, 10, 50], ids=["nodiscount", "10pct", "50pct"])
def test_discount_combinations(price, pct):
    result = apply_discount(price, pct)
    assert result == round(price * (1 - pct/100), 2)
    assert result <= price

# This creates 3x3=9 test combinations automatically when run via pytest
print("Stacked parametrize creates the cartesian product of both parameter sets")
for price in [100, 50, 200]:
    for pct in [0, 10, 50]:
        print(f"  price={price}, pct={pct}% -> {apply_discount(price, pct)}")

### 17. Parametrize with pytest.param for Marking Individual Cases

Use `pytest.param(..., marks=pytest.mark.xfail)` to mark a SPECIFIC parametrized case as expected-to-fail, while others run normally.

In [ ]:
import pytest

def divide(a, b):
    return a / b

@pytest.mark.parametrize("a,b,expected", [
    (10, 2, 5.0),
    (9, 3, 3.0),
    pytest.param(5, 0, None, marks=pytest.mark.xfail(raises=ZeroDivisionError)),
])
def test_divide(a, b, expected):
    assert divide(a, b) == expected

# Demonstrating manually
for a, b, expected in [(10, 2, 5.0), (9, 3, 3.0)]:
    print(f"divide({a},{b}) = {divide(a,b)} (expected {expected})")
try:
    divide(5, 0)
except ZeroDivisionError:
    print("divide(5,0) raised ZeroDivisionError as expected (xfail case)")

### 18. Basic Mock with return_value

Create a `Mock` simulating an external API client, configuring its `return_value`, and verify it was called correctly.

In [ ]:
from unittest.mock import Mock

api_client = Mock()
api_client.fetch_user.return_value = {"id": 1, "name": "Alice"}

result = api_client.fetch_user(user_id=1)
print(result)

api_client.fetch_user.assert_called_once_with(user_id=1)
print("Verified: called exactly once with the right arguments")

### 19. Mock with side_effect Raising Exceptions

Configure a Mock's `side_effect` to raise an exception, simulating a failing dependency, and verify your code handles it.

In [ ]:
from unittest.mock import Mock

db = Mock()
db.save.side_effect = ConnectionError("Database unreachable")

def save_record(db, record):
    try:
        db.save(record)
        return "saved"
    except ConnectionError as e:
        return f"failed: {e}"

result = save_record(db, {"id": 1})
print(result)

### 20. Mock with side_effect as a List (Sequential Returns)

Configure `side_effect` as a LIST so each call returns the next value in sequence — useful for simulating retry scenarios.

In [ ]:
from unittest.mock import Mock

flaky_service = Mock()
flaky_service.call.side_effect = [ConnectionError("fail 1"), ConnectionError("fail 2"), "success!"]

def call_with_retry(service, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return service.call()
        except ConnectionError as e:
            print(f"  Attempt {attempt+1} failed: {e}")
    raise RuntimeError("All attempts exhausted")

result = call_with_retry(flaky_service)
print("Final result:", result)

### 21. MagicMock for Magic Method Support

Show that `MagicMock` (unlike plain `Mock`) supports dunder methods like `__len__`, `__getitem__` out of the box.

In [ ]:
from unittest.mock import MagicMock, Mock

magic = MagicMock()
magic.__len__.return_value = 5
magic.__getitem__.side_effect = lambda i: f"item-{i}"

print(len(magic))
print(magic[2])

plain = Mock()
try:
    len(plain)    # plain Mock does NOT support magic methods by default
except TypeError as e:
    print(f"Plain Mock fails: {e}")

### 22. patch() as Decorator with Multiple Mocks

Use `@patch` TWICE (stacked) to mock two different dependencies in one test function, noting the bottom-up argument order.

In [ ]:
from unittest.mock import patch

def get_weather_report(weather_api, translator):
    raw = weather_api.get_temp()
    return translator.translate(f"{raw} degrees")

@patch("__main__.translator_stub")
@patch("__main__.weather_api_stub")
def test_weather_report(mock_weather, mock_translator):
    mock_weather.get_temp.return_value = 72
    mock_translator.translate.return_value = "72 degrees (translated)"
    result = get_weather_report(mock_weather, mock_translator)
    assert result == "72 degrees (translated)"
    print("Test passed:", result)

weather_api_stub = None
translator_stub = None
test_weather_report()

### 23. patch.object for Patching a Specific Method on a Real Class

Use `patch.object(SomeClass, 'method_name')` to mock just ONE method of a real class while leaving the rest of the class intact.

In [ ]:
from unittest.mock import patch

class EmailService:
    def send(self, to, subject):
        raise RuntimeError("Would actually send a real email!")
    def validate_address(self, address):
        return "@" in address

service = EmailService()
print("Real method still works:", service.validate_address("test@example.com"))

with patch.object(EmailService, "send", return_value="mocked send successful"):
    result = service.send("test@example.com", "Hello")
    print("Patched method result:", result)

print("After patch context, real method restored:")
try:
    service.send("test@example.com", "Hello")
except RuntimeError as e:
    print(f"  Real method raises again: {e}")

### 24. Mock with spec= for Interface-Accurate Mocking

Use `Mock(spec=RealClass)` so the mock only allows attributes/methods that ACTUALLY exist on the real class — catching typos in test code.

In [ ]:
from unittest.mock import Mock

class PaymentGateway:
    def charge(self, amount): ...
    def refund(self, transaction_id): ...

mock_gateway = Mock(spec=PaymentGateway)
mock_gateway.charge(100)             # fine, charge() is a real method
print("charge() call recorded:", mock_gateway.charge.called)

try:
    mock_gateway.nonexistent_method()    # spec catches this typo immediately!
except AttributeError as e:
    print(f"Caught typo via spec: {e}")

### 25. call_args, call_args_list, and call_count Inspection

Call a mock MULTIPLE times with different arguments, then inspect `call_count`, `call_args` (most recent), and `call_args_list` (all calls).

In [ ]:
from unittest.mock import Mock

logger = Mock()
logger.log("INFO", "Starting up")
logger.log("WARNING", "Low memory")
logger.log("ERROR", "Connection failed")

print("Call count:", logger.log.call_count)
print("Most recent call:", logger.log.call_args)
print("All calls:")
for call in logger.log.call_args_list:
    print(" ", call)

# Assert something specific happened among ALL calls
all_levels = [call.args[0] for call in logger.log.call_args_list]
assert "ERROR" in all_levels
print("\nConfirmed an ERROR was logged at some point")

### 26. Testing Async Functions with pytest-style Coroutines

Write and directly await an async test function validating an async service call — demonstrating the pattern used with `pytest-asyncio` (executed here directly).

In [ ]:
import asyncio
from unittest.mock import AsyncMock

async def fetch_user_async(client, user_id):
    return await client.get_user(user_id)

async def test_async_function():
    mock_client = AsyncMock()
    mock_client.get_user.return_value = {"id": 1, "name": "Alice"}

    result = await fetch_user_async(mock_client, 1)
    assert result == {"id": 1, "name": "Alice"}
    mock_client.get_user.assert_awaited_once_with(1)
    print("Async test passed:", result)

await test_async_function()

### 27. Testing a Service Layer: Mocking Multiple Collaborators

Build a small `OrderProcessor` depending on `inventory`, `payment`, and `notifier` collaborators, then write a full test mocking ALL three to isolate the logic under test.

In [ ]:
from unittest.mock import Mock

class OrderProcessor:
    def __init__(self, inventory, payment, notifier):
        self.inventory, self.payment, self.notifier = inventory, payment, notifier

    def process(self, order):
        if not self.inventory.is_available(order["sku"]):
            return {"success": False, "reason": "out_of_stock"}
        charge = self.payment.charge(order["amount"])
        if not charge["success"]:
            return {"success": False, "reason": "payment_failed"}
        self.inventory.reserve(order["sku"])
        self.notifier.send(order["customer_email"], "Order confirmed")
        return {"success": True}

inventory = Mock()
payment = Mock()
notifier = Mock()
inventory.is_available.return_value = True
payment.charge.return_value = {"success": True}

processor = OrderProcessor(inventory, payment, notifier)
result = processor.process({"sku": "ABC", "amount": 50, "customer_email": "a@b.com"})

assert result["success"] is True
inventory.reserve.assert_called_once_with("ABC")
notifier.send.assert_called_once()
print("Full service-layer test passed with all 3 collaborators mocked:", result)

### 28. Negative-Path Test: Verifying Something Did NOT Happen

Extending the previous pattern, write a test for the FAILURE path proving that `notifier.send` and `inventory.reserve` are correctly NEVER called when payment fails.

In [ ]:
from unittest.mock import Mock

class OrderProcessor:
    def __init__(self, inventory, payment, notifier):
        self.inventory, self.payment, self.notifier = inventory, payment, notifier
    def process(self, order):
        if not self.inventory.is_available(order["sku"]):
            return {"success": False, "reason": "out_of_stock"}
        charge = self.payment.charge(order["amount"])
        if not charge["success"]:
            return {"success": False, "reason": "payment_failed"}
        self.inventory.reserve(order["sku"])
        self.notifier.send(order["customer_email"], "Order confirmed")
        return {"success": True}

inventory = Mock()
payment = Mock()
notifier = Mock()
inventory.is_available.return_value = True
payment.charge.return_value = {"success": False}   # payment FAILS this time

processor = OrderProcessor(inventory, payment, notifier)
result = processor.process({"sku": "ABC", "amount": 50, "customer_email": "a@b.com"})

assert result["success"] is False
inventory.reserve.assert_not_called()      # critical negative assertion!
notifier.send.assert_not_called()           # critical negative assertion!
print("Negative-path test passed:", result)
print("Confirmed inventory was NOT reserved and customer was NOT notified on failure")